# Detecção de Landmarks (Keypoints)

Nesta aula, abordaremos a **Detecção de Landmarks**, também conhecida como detecção de keypoints. O objetivo é prever as coordenadas $(x, y)$ de pontos de interesse específicos em uma imagem, como cantos dos olhos, ponta do nariz e contornos da boca em rostos humanos.

Diferente da detecção de objetos (onde prevemos *bounding boxes*), na detecção de landmarks prevemos pontos exatos. Para um rosto com $K$ landmarks, nosso modelo deve produzir uma saída de tamanho $2K$ (um $x$ e um $y$ para cada ponto). 

Neste dataset, temos 15 keypoints faciais ($K=15$), resultando em 30 valores a serem previstos. Como os pontos representam coordenadas em uma imagem, estamos lidando com um problema de **Regressão**.

A função de perda padrão para regressão é o Erro Médio Quadrático (MSE):
$$ Loss = \frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{2K} (\hat{y}_{ij} - y_{ij})^2 $$
onde $\hat{y}$ são as coordenadas previstas e $y$ são as coordenadas reais.

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gdown
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Dataset

O dataset é fornecido em um arquivo CSV. Cada linha possui as coordenadas dos 15 keypoints e a última coluna contém os pixels da imagem de 96x96 como uma string de números separados por espaço.

Alguns keypoints podem estar faltando (representados por campos vazios). Para simplificar, vamos preencher os valores ausentes com a média ou removê-los, mas nossa abordagem inicial será usar `fillna` ou ignorar colunas.


In [ ]:
gdown.download("https://drive.google.com/uc?id=14DTluCyfSr7MPkFaFVgbGn-Olm0zFAdK", "facial-keypoints-dataset.zip", quiet=False)

with zipfile.ZipFile("facial-keypoints-dataset.zip") as z:
    z.extractall("data")

In [ ]:
class FacialKeypointsDataset(Dataset):
    def __init__(self, data_df, transform=None):
        self.data = data_df.dropna()
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        img_str = row['Image']
        img = np.array([int(p) for p in img_str.split()]).reshape(96, 96).astype(np.float32)
        keypoints = row.drop('Image').values.astype(np.float32)
        
        img = img / 255.0
        keypoints = keypoints / 96.0
        
        img = torch.tensor(img).unsqueeze(0) # [1, 96, 96]
        keypoints = torch.tensor(keypoints)  # [30]
        
        if self.transform:
            img = self.transform(img)
            
        return img, keypoints

In [ ]:
# Lê o CSV único e divide em treino e teste
df = pd.read_csv('data/facial-keypoints-dataset.csv')
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_dataset = FacialKeypointsDataset(train_df)
test_dataset = FacialKeypointsDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

imgs, targets = next(iter(train_loader))
print(f'Shape Imagens: {imgs.shape} | Shape Targets: {targets.shape}')

In [ ]:
def show_keypoints(img, keypoints):
    plt.imshow(img.squeeze(), cmap='gray')
    # O target foi normalizado para [0, 1], desfazemos para plotar no tamanho 96x96
    keypoints = keypoints * 96.0
    for i in range(0, len(keypoints), 2):
        plt.scatter(keypoints[i], keypoints[i+1], s=10, marker='.', c='r')

plt.figure(figsize=(10, 4))
for i in range(4):
    plt.subplot(1, 4, i+1)
    show_keypoints(imgs[i].numpy(), targets[i].numpy())
    plt.axis('off')
plt.tight_layout()
plt.show()

## Arquitetura da CNN

Vamos construir uma Rede Neural Convolucional simples que extrai características da imagem 96x96 e usa camadas densas no final para regredir as 30 coordenadas.


In [ ]:
class LandmarkCNN(nn.Module):
    def __init__(self, num_points=15):
        super().__init__()
        self.num_points = num_points
        
        self.features = nn.Sequential(
            # Entrada: [Batch, 1, 96, 96]
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Saída: [Batch, 32, 48, 48]
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Saída: [Batch, 64, 24, 24]
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Saída: [Batch, 128, 12, 12]
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Linear(512, self.num_points * 2), # N pontos x 2 coordenadas
            nn.Sigmoid() # Restringe saída a [0, 1] visto que nossos targets foram normalizados
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Preserva o batch e faz flatten (Batch, 128*12*12)
        x = self.regressor(x)
        return x

In [ ]:
model = LandmarkCNN(num_points=15).to(device)
print(model)

## Treinamento

Usaremos MSELoss e Adam optimizer.


In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 15
history = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    loop = tqdm(train_loader, leave=False)
    for imgs, targets in loop:
        imgs, targets = imgs.to(device), targets.to(device)
        
        predictions = model(imgs)
        loss = criterion(predictions, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_description(f'Epoch [{epoch+1}/{num_epochs}]')
        loop.set_postfix(loss=loss.item())
        
    history.append(epoch_loss / len(train_loader))
    print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {history[-1]:.6f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history, label='Train Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.title('Curva de Treinamento')
plt.show()

## Avaliação e Visualização

Vamos plotar as predições do modelo para verificarmos a eficácia no conjunto de teste.


In [ ]:
model.eval()
imgs, targets = next(iter(test_loader))

with torch.no_grad():
    predictions = model(imgs.to(device)).cpu()

plt.figure(figsize=(16, 4))
for i in range(8):
    if i >= len(imgs):
        break
    plt.subplot(2, 4, i+1)
    # Verde para o real, Vermelho para a predição
    plt.imshow(imgs[i].squeeze(), cmap='gray')
    
    real_pts = targets[i].numpy() * 96.0
    pred_pts = predictions[i].numpy() * 96.0
    
    for j in range(0, model.num_points * 2, 2):
        plt.scatter(real_pts[j], real_pts[j+1], s=10, marker='.', c='g') # Real
        plt.scatter(pred_pts[j], pred_pts[j+1], s=10, marker='.', c='r') # Previsto
        
    plt.axis('off')

plt.tight_layout()
plt.show()

## Exercícios

### Exercício 1: Data Augmentation
Transformações como rotação e flip podem ajudar a generalizar o modelo. No entanto, se você rotacionar a imagem, as coordenadas $x, y$ originais ficarão incorretas. Como você lidaria com data augmentation em landmarks?.

### Exercício 2: Melhorias na Arquitetura
A arquitetura atual é básica. Implemente a utilização de `BatchNorm2d` nas camadas convolucionais ou utilize técnicas de Dropout no classificador final e compare o erro final após 15 épocas.